In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis, ks_2samp
from factor_model import FactorModel, reconstruct_returns, load_model
from factor_diffusion_train import *
from factor_diffusion_sample import *

In [ ]:
train_model = load_model(f"{PREFIX}")
test_model = load_model(f"{PREFIX}/test")
print(PREFIX)

In [ ]:
R_df = pd.read_parquet(train_model.data_source)
R_testdf = pd.read_parquet(test_model.data_source)
common_stocks = np.intersect1d(R_df['csecid'].unique(), R_testdf['csecid'].unique())
col_indices = np.searchsorted(R_df['csecid'].values, common_stocks)
sorted_stocks = np.sort(R_df['csecid'].unique())
col_indices = np.searchsorted(np.sort(R_df['csecid'].unique()), common_stocks)
R_testdf = R_testdf[R_testdf['csecid'].isin(common_stocks)]
R_df = R_df[R_df['csecid'].isin(common_stocks)]

In [ ]:
R_train = R_df.pivot_table(index="date", columns="csecid", values="returns").values
rng = np.random.default_rng(42)
row_idx = rng.choice(R_train.shape[0], size=NUM_GENERATE, replace=False)
R_rs = R_train[row_idx, :]
R_test = R_testdf.pivot_table(index="date", columns="csecid", values="returns").values
print(f"Train returns: {R_train.shape}  NaN fraction: {np.isnan(R_train).mean():.2%}")
print(f"Test returns:  {R_test.shape}   NaN fraction: {np.isnan(R_test).mean():.2%}")
print(f"Resample returns: {R_rs.shape}  NaN fraction: {np.isnan(R_rs).mean():.2%}")

In [ ]:

sampled_factors = np.load(f"{PREFIX}/samples/factor_{MODE}_{NUM_GENERATE}.npy")
fs_full = np.column_stack([
    np.ones((len(sampled_factors), 1), dtype=np.float32),
    sampled_factors.astype(np.float32),
])
R_gen = reconstruct_returns(train_model, fs_full)[:, col_indices]
print(f"Generated: {R_gen.shape}")

In [ ]:
## 3.1 Stock-Level Moment Scatter Plots
from scipy.stats import pearsonr

MOMENTS = {
    "Mean":     lambda X: np.nanmean(X, axis=0),
    "Std":      lambda X: np.nanstd(X, axis=0),
    "Skewness": lambda X: skew(X, axis=0, nan_policy="omit"),
    "Kurtosis": lambda X: kurtosis(X, axis=0, nan_policy="omit"),
}

m_actual_tr  = {k: fn(R_train) for k, fn in MOMENTS.items()}
m_actual_oos = {k: fn(R_test)  for k, fn in MOMENTS.items()}
m_gen        = {k: fn(R_gen)   for k, fn in MOMENTS.items()}
m_rs         = {k: fn(R_rs)    for k, fn in MOMENTS.items()}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for col, (mname, _) in enumerate(MOMENTS.items()):
    for row, (actual_label, m_actual) in enumerate([
            ("Train", m_actual_tr),
            ("OOS",   m_actual_oos)]):
        ax  = axes[row, col]
        x   = m_actual[mname]
        y_g = m_gen[mname]
        y_r = m_rs[mname]

        mask_g = np.isfinite(x) & np.isfinite(y_g)
        mask_r = np.isfinite(x) & np.isfinite(y_r)

        all_vals = np.concatenate([x[mask_g], y_g[mask_g],
                                   x[mask_r], y_r[mask_r]])
        lo = np.percentile(all_vals, 1)
        hi = np.percentile(all_vals, 99)

        ax.scatter(x[mask_g], y_g[mask_g], s=5, alpha=0.25, linewidths=0,
                   color="#c44e52", label="Generated")
        ax.scatter(x[mask_r], y_r[mask_r], s=5, alpha=0.25, linewidths=0,
                   color="#2ca02c", label="Resample")
        ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_xlabel(f"Actual ({actual_label})", fontsize=9)
        ax.set_title(mname, fontsize=10)
        ax.set_aspect("equal")
        ax.legend(fontsize=6, markerscale=2, loc="upper left")

for row, label in enumerate(["Train", "OOS"]):
    axes[row, 0].annotate(label, xy=(-0.22, 0.5), xycoords="axes fraction",
                          fontsize=11, rotation=90, va="center", fontweight="bold")

fig.suptitle("3.1  Stock-Level Moment Scatter Plots", fontsize=13)
fig.tight_layout()
plt.show()

# ── Pearson R² ────────────────────────────────────────────────────────────────
col_w = 14
header = f"{'':10}" + "".join(f"{m:<{col_w}}" for m in MOMENTS)
print("Pearson R² (Generated vs Actual, per-stock moments):")
print(header)
print("-" * len(header))
for actual_label, m_actual in [("Train", m_actual_tr), ("OOS", m_actual_oos)]:
    row = f"{actual_label:<10}"
    for mname in MOMENTS:
        x, y = m_actual[mname], m_gen[mname]
        mask = np.isfinite(x) & np.isfinite(y)
        r, _ = pearsonr(x[mask], y[mask])
        row += f"{r**2:<{col_w}.4f}"
    print(row)


In [ ]:
## 3.2 Distribution of KS Test P-values
pvals_gen_train, pvals_gen_oos = [], []
pvals_rs_train,  pvals_rs_oos  = [], []
S = R_train.shape[1]

for s in range(S):
    g   = R_gen[:, s];   g   = g[np.isfinite(g)]
    rs  = R_rs[:, s];    rs  = rs[np.isfinite(rs)]
    tr  = R_train[:, s]; tr  = tr[np.isfinite(tr)]
    oos = R_test[:, s];  oos = oos[np.isfinite(oos)]
    if len(g)  > 5 and len(tr)  > 5: pvals_gen_train.append(ks_2samp(g,  tr).pvalue)
    if len(g)  > 5 and len(oos) > 5: pvals_gen_oos.append(  ks_2samp(g,  oos).pvalue)
    if len(rs) > 5 and len(tr)  > 5: pvals_rs_train.append( ks_2samp(rs, tr).pvalue)
    if len(rs) > 5 and len(oos) > 5: pvals_rs_oos.append(   ks_2samp(rs, oos).pvalue)

pvals_gen_train = np.array(pvals_gen_train)
pvals_gen_oos   = np.array(pvals_gen_oos)
pvals_rs_train  = np.array(pvals_rs_train)
pvals_rs_oos    = np.array(pvals_rs_oos)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

for ax, pv, label in [(ax1, pvals_gen_train, "Generated vs Train"),
                      (ax2, pvals_gen_oos,   "Generated vs OOS")]:
    n_bins = 20
    ax.hist(pv, bins=n_bins, range=(0, 1), color="#1a6faf", alpha=0.75, edgecolor="white")
    ax.axhline(len(pv) / n_bins, color="red", linestyle="--", lw=1.2, label="Uniform")
    ax.set_xlabel("KS p-value", fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.set_title(f"3.2  KS P-values: {label}", fontsize=10)
    ax.legend(fontsize=8)
    frac = (pv < 0.05).mean()
    ax.text(0.97, 0.95, f"p<0.05: {frac:.1%}", transform=ax.transAxes,
            ha="right", va="top", fontsize=9)

fig.tight_layout()
plt.show()

# ── KS p-value summary: Generated vs Resample ────────────────────────────────
col_w = 20
header = f"{'':16}" + f"{'Generated':<{col_w}}" + f"{'Resample':<{col_w}}"
print("KS p-value Summary (per-stock, vs Train / OOS):")
print(header)
print("-" * len(header))
for label, pv_g, pv_r in [
    ("vs Train p>0.05", pvals_gen_train, pvals_rs_train),
    ("vs Train median", pvals_gen_train, pvals_rs_train),
    ("vs OOS   p>0.05", pvals_gen_oos,   pvals_rs_oos),
    ("vs OOS   median", pvals_gen_oos,   pvals_rs_oos),
]:
    if "p>0.05" in label:
        gen_val = f"{(pv_g > 0.05).mean():.1%}"
        rs_val  = f"{(pv_r > 0.05).mean():.1%}"
    else:
        gen_val = f"{np.median(pv_g):.4f}"
        rs_val  = f"{np.median(pv_r):.4f}"
    print(f"{label:<16}{gen_val:<{col_w}}{rs_val:<{col_w}}")


In [ ]:
beta_c    = train_model.beta[:, col_indices]
res_std_c = train_model.res_std[col_indices]

test_full = pd.read_parquet(test_model.data_source)
col_indices_test = np.searchsorted(np.sort(test_full['csecid'].unique()), common_stocks)

R_train_cs = R_df.pivot_table(index="date", columns="csecid", values="returns").values
sys_train  = train_model.F.values.astype(np.float32) @ beta_c
idio_train = train_model.residuals[:, col_indices].astype(np.float32)

R_oos_cs = R_testdf.pivot_table(index="date", columns="csecid", values="returns").values
sys_oos  = test_model.F.values.astype(np.float32) @ beta_c
idio_oos = test_model.residuals[:, col_indices_test].astype(np.float32)

sys_gen  = fs_full @ beta_c
t_noise = np.random.default_rng(42).standard_t(train_model.res_df, size=sys_gen.shape).astype(np.float32)
idio_gen = t_noise * np.sqrt((train_model.res_df - 2) / train_model.res_df) * res_std_c
R_gen_cs = sys_gen + idio_gen

panels = [
    ("Total",         R_gen_cs, R_oos_cs,  R_train_cs),
    ("Systematic",    sys_gen,  sys_oos,   sys_train),
    ("Idiosyncratic", idio_gen, idio_oos,  idio_train),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (comp, gmat, omat, tmat) in zip(axes, panels):
    gv = np.nanstd(gmat, axis=1)
    ov = np.nanstd(omat, axis=1)
    tv = np.nanstd(tmat, axis=1)
    ax.hist(tv, bins=50, density=True, alpha=0.60, color="#2ca02c", label="Train")
    ax.hist(ov, bins=50, density=True, alpha=0.60, color="#1a6faf", label="OOS")
    ax.hist(gv, bins=50, density=True, alpha=0.60, color="#c44e52", label="Generated")
    for val, col in [(tv.mean(), "#2ca02c"), (ov.mean(), "#1a6faf"), (gv.mean(), "#c44e52")]:
        ax.axvline(val, color=col, linestyle="--", lw=1.2)
    r_gen = gv.mean() / (ov.mean() + 1e-12)
    r_tr  = tv.mean() / (ov.mean() + 1e-12)
    ax.set_xlabel("Cross-Sectional Std", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_title(comp + "\nGen/OOS=%.2f  Train/OOS=%.2f" % (r_gen, r_tr), fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle("4.  Cross-Sectional Dispersion & Decomposition", fontsize=12)
fig.tight_layout()
plt.show()

# ── Wasserstein-1: Generated vs OOS, Resample vs OOS ────────────────────────
from scipy.stats import wasserstein_distance as w1dist

row_idx_r = row_idx[row_idx < len(sys_train)]
sys_rs    = sys_train[row_idx_r]
idio_rs   = idio_train[row_idx_r]

rs_panels = [
    ("Total",         R_rs,    R_oos_cs),
    ("Systematic",    sys_rs,  sys_oos),
    ("Idiosyncratic", idio_rs, idio_oos),
]

col_w = 20
header = f"{'Component':<16}{'W1(Gen, OOS)':<{col_w}}{'W1(Resample, OOS)':<{col_w}}"
print("Wasserstein-1 Distance of Cross-Sectional Std (vs OOS):")
print(header)
print("-" * len(header))
for (comp, gmat, omat, _), (_, rsmat, _) in zip(panels, rs_panels):
    gv  = np.nanstd(gmat,  axis=1)
    ov  = np.nanstd(omat,  axis=1)
    rsv = np.nanstd(rsmat, axis=1)
    print(f"{comp:<16}{w1dist(gv, ov):<{col_w}.4f}{w1dist(rsv, ov):<{col_w}.4f}")
